In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_6.py ──
"""
Shared infrastructure for MLFP04 Exercise 6 — NLP & Topic Modelling.

Contains: corpus loading, TF-IDF/CountVectorizer construction, NPMI
coherence scoring, sentiment lexicons, Singapore/APAC text-analytics
scenario helpers, and plot output directory management.

Technique-specific code (TF-IDF/BM25 scoring, NMF decomposition, LDA
fitting, BERTopic pipeline, Word2Vec sentiment classifier) does NOT
belong here — it lives in the per-technique files under
modules/mlfp04/solutions/ex_6/.
"""

from collections import Counter
from pathlib import Path
from typing import Iterable

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex6_nlp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# TOY CORPUS — small Singapore-flavoured corpus for teaching derivations
# ════════════════════════════════════════════════════════════════════════

TOY_CORPUS: list[str] = [
    "Singapore economy grew strongly in 2024",
    "Singapore property market shows resilience",
    "MAS tightens monetary policy amid global uncertainty",
    "Property developers report strong demand",
    "Singapore government announces new housing measures",
    "Global markets react to central bank decisions",
    "Technology sector leads Singapore stock exchange",
    "Housing prices continue upward trend in Singapore",
]


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore/APAC document corpus
# ════════════════════════════════════════════════════════════════════════


def load_corpus(min_chars: int = 100) -> pl.DataFrame:
    """Load the MLFP document corpus (title + content + category).

    Returns a polars DataFrame with a derived 'text' column that concatenates
    title and content, filtered to rows with content longer than ``min_chars``.
    Categories include singapore_economy, singapore_housing, asean_economy,
    ml_fundamentals, and several other topical clusters — well-suited to
    unsupervised topic discovery.
    """
    loader = MLFPDataLoader()
    df = loader.load("mlfp03", "documents.parquet")
    return df.with_columns(
        (pl.col("title") + ". " + pl.col("content")).alias("text"),
    ).filter(pl.col("content").str.len_chars() > min_chars)


def corpus_as_lists(
    df: pl.DataFrame,
) -> tuple[list[str], list[str]]:
    """Split a corpus frame into parallel (documents, categories) lists."""
    return df["text"].to_list(), df["category"].to_list()


# ════════════════════════════════════════════════════════════════════════
# NPMI TOPIC COHERENCE
# ════════════════════════════════════════════════════════════════════════


def compute_npmi(
    documents: list[str],
    topic_words: list[list[str]],
    max_docs: int = 3000,
) -> list[float]:
    """Compute Normalised Pointwise Mutual Information coherence per topic.

    NPMI(w_i, w_j) = log(P(w_i, w_j) / (P(w_i) P(w_j))) / (-log P(w_i, w_j))

    Range is [-1, 1]. Higher = more coherent topic. We approximate
    probabilities from a document-level co-occurrence count over the first
    ``max_docs`` documents (enough for a stable signal).
    """
    docs = documents[:max_docs]
    word_doc_count: Counter[str] = Counter()
    pair_doc_count: Counter[tuple[str, str]] = Counter()
    n_docs = len(docs)

    for doc in docs:
        words = set(doc.lower().split())
        for w in words:
            word_doc_count[w] += 1
        word_list = list(words)
        for i in range(len(word_list)):
            for j in range(i + 1, len(word_list)):
                pair = tuple(sorted([word_list[i], word_list[j]]))
                pair_doc_count[pair] += 1

    coherences: list[float] = []
    for topic in topic_words:
        npmi_sum = 0.0
        n_pairs = 0
        for i in range(len(topic)):
            for j in range(i + 1, len(topic)):
                w_i, w_j = topic[i].lower(), topic[j].lower()
                pair = tuple(sorted([w_i, w_j]))
                p_i = word_doc_count.get(w_i, 0) / max(n_docs, 1)
                p_j = word_doc_count.get(w_j, 0) / max(n_docs, 1)
                p_ij = pair_doc_count.get(pair, 0) / max(n_docs, 1)
                if p_ij > 0 and p_i > 0 and p_j > 0:
                    pmi = np.log(p_ij / (p_i * p_j))
                    npmi = pmi / (-np.log(p_ij))
                    npmi_sum += npmi
                    n_pairs += 1
        coherences.append(npmi_sum / max(n_pairs, 1))
    return coherences


# ════════════════════════════════════════════════════════════════════════
# SENTIMENT LEXICONS — baseline for review triage
# ════════════════════════════════════════════════════════════════════════

POSITIVE_WORDS: frozenset[str] = frozenset(
    {
        "good",
        "great",
        "excellent",
        "best",
        "strong",
        "growth",
        "resilience",
        "success",
        "positive",
        "improved",
        "innovative",
        "robust",
        "outperformed",
        "gains",
        "upbeat",
    }
)

NEGATIVE_WORDS: frozenset[str] = frozenset(
    {
        "bad",
        "poor",
        "decline",
        "fall",
        "weak",
        "crisis",
        "risk",
        "loss",
        "negative",
        "failed",
        "uncertainty",
        "downturn",
        "slump",
        "recession",
        "worry",
    }
)


def lexicon_sentiment(docs: Iterable[str]) -> np.ndarray:
    """Score each document in ``docs`` as a scalar in [-1, 1] via a naive lexicon.

    (pos - neg) / (pos + neg), zero when neither list matches.
    """
    scores: list[float] = []
    for doc in docs:
        words = set(doc.lower().split())
        pos = len(words & POSITIVE_WORDS)
        neg = len(words & NEGATIVE_WORDS)
        total = pos + neg
        scores.append((pos - neg) / total if total > 0 else 0.0)
    return np.asarray(scores, dtype=np.float64)


# ════════════════════════════════════════════════════════════════════════
# SCENARIO HELPERS — Singapore / APAC text analytics context
# ════════════════════════════════════════════════════════════════════════

SCENARIOS: dict[str, str] = {
    "tfidf_bm25": (
        "CASE: ST Engineering internal document search. 180K engineering "
        "reports across aerospace, marine, and electronics business units. "
        "TF-IDF vs BM25 determines which report a manager sees first when "
        "searching 'turbine blade fatigue.' A missed report costs ~S$450K "
        "in duplicated R&D work. BM25's length normalisation (b=0.75) "
        "ensures short executive summaries compete fairly with 80-page "
        "technical manuals, improving first-hit accuracy by ~14%."
    ),
    "nmf_topics": (
        "CASE: Singapore Press Holdings (SPH) newsroom content tagging. "
        "~2,400 articles/day across The Straits Times, Business Times, "
        "zaobao.com. NMF runs nightly on the 24-hour TF-IDF matrix and "
        "produces 20 interpretable topics (housing, MAS monetary policy, "
        "SEA Games, etc.) for the recommendation engine. Non-negativity "
        "means the topic-keyword report is directly auditable by the "
        "editorial desk. Ad yield on tagged articles is +11% vs untagged."
    ),
    "lda_topics": (
        "CASE: Monetary Authority of Singapore (MAS) enforcement scanning. "
        "~60K complaint emails/year across retail banking, insurance, "
        "fintech. LDA's mixed-membership model matters — a single complaint "
        "about a crypto rug pull mentioning SGD withdrawals touches BOTH "
        "a 'digital asset fraud' topic AND a 'cross-border payments' topic. "
        "Soft topic assignments route the complaint to both enforcement "
        "teams, cutting median resolution time from 18 days to 11."
    ),
    "bertopic": (
        "CASE: Grab customer-support ticket clustering across Singapore, "
        "Indonesia, Thailand, Vietnam. ~35K tickets/week, mixed English + "
        "Bahasa Indonesia + Thai + Vietnamese. BERTopic's multilingual "
        "sentence-transformer embeddings discover topics that TF-IDF "
        "misses ('driver helmet policy' clusters across languages because "
        "the embeddings are language-agnostic). NPMI coherence of 0.18 vs "
        "LDA's 0.08; each well-clustered ticket saves ~S$3.20 in routing."
    ),
    "sentiment_word2vec": (
        "CASE: DBS Bank app-store review triage. 40K reviews/month across "
        "iOS, Google Play, and Huawei AppGallery in English, Mandarin, "
        "Malay, Tamil. A Word2Vec + logistic-regression sentiment classifier "
        "trained on labelled English reviews transfers to the other three "
        "languages via shared subword tokens. Each negative review routed "
        "to CX within 10 min prevents ~S$8K of avoided viral complaints — "
        "catching 20 extra/month is S$160K vs S$120 in compute."
    ),
}


def print_scenario(name: str) -> None:
    """Print a named Singapore/APAC scenario block for the APPLY phase."""
    body = SCENARIOS.get(name, "")
    if not body:
        return
    print("\n" + "=" * 70)
    print(f"  APPLY — {name}")
    print("=" * 70)
    print(body)
    print("=" * 70 + "\n")




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 6.1: TF-IDF and BM25 — Classic Text Retrieval
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Derive TF-IDF from a bag-of-words matrix
#   - Implement BM25 with saturation (k1) and length normalisation (b)
#   - Compare TF-IDF and BM25 on the same corpus
#   - Apply the technique to ST Engineering internal document search
#
# PREREQUISITES: Linear algebra (sparse matrices), basic probability.
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — why IDF exists and what BM25 fixes
#   2. Build — CountVectorizer -> TfidfVectorizer -> manual BM25
#   3. Train — score terms across the corpus
#   4. Visualise — top-term rankings, BM25 saturation curve
#   5. Apply — ST Engineering scenario
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from kailash_ml import ModelVisualizer



## TASK 2 — BUILD: bag-of-words -> TF-IDF


In [ ]:
print("=" * 70)
print("  Bag-of-Words Warmup")
print("=" * 70)

# TODO: Build a CountVectorizer with English stop words and fit_transform
# the TOY_CORPUS. Store the fitted vectorizer and the matrix.
# Hint: CountVectorizer(stop_words="english"), .fit_transform(TOY_CORPUS)
bow_vectorizer = ____
X_bow = ____
bow_vocab = bow_vectorizer.get_feature_names_out()

print(f"Vocabulary size: {len(bow_vocab)}")
print(f"Matrix shape:    {X_bow.shape}")


print("\n" + "=" * 70)
print("  TF-IDF Weights")
print("=" * 70)

# TODO: Build a TfidfVectorizer (stop_words="english", norm="l2") and
# fit_transform the TOY_CORPUS. Expose get_feature_names_out() and idf_.
tfidf_vectorizer = ____
X_tfidf = ____
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()
idf_values = tfidf_vectorizer.idf_



### Checkpoint 1


In [ ]:
idf_dict = dict(zip(tfidf_vocab, idf_values))
assert (
    idf_dict["singapore"] < idf_dict["monetary"]
), "Task 2: 'singapore' (common) should have lower IDF than 'monetary' (rare)"
row_norms = np.sqrt(np.asarray(X_tfidf.multiply(X_tfidf).sum(axis=1)).flatten())
assert all(
    abs(n - 1.0) < 0.01 for n in row_norms if n > 0
), "Task 2: TF-IDF rows should be L2-normalised"
print("\n[ok] Checkpoint 1 passed — TF-IDF IDF correctly ranks rare vs common terms\n")



## TASK 3 — TRAIN-FREE SCORING: manual BM25


Return BM25 score for a single (term, document) pair.


In [ ]:
# BM25(t, d) = IDF(t) * (tf * (k1 + 1)) /
#              (tf + k1 * (1 - b + b * |d| / avgdl))


def bm25_score(
    tf: float,
    df: int,
    N: int,
    dl: int,
    avgdl: float,
    k1: float = 1.2,
    b: float = 0.75,
) -> float:
    # TODO: Compute the Robertson/Sparck-Jones IDF:
    # idf = log((N - df + 0.5) / (df + 0.5) + 1)
    idf = ____

    # TODO: Compute the saturated TF component using tf, k1, b, dl, avgdl
    # Hint: numerator = tf * (k1 + 1); denominator = tf + k1 * (1 - b + b * dl / avgdl)
    tf_component = ____

    return idf * tf_component


doc_lengths = [len(doc.split()) for doc in TOY_CORPUS]
avgdl = float(np.mean(doc_lengths))
N_docs = len(TOY_CORPUS)
sg_df = sum(1 for doc in TOY_CORPUS if "singapore" in doc.lower())

tf_grid = np.arange(1, 21)
saturation_scores = [
    bm25_score(float(tf), df=2, N=100, dl=50, avgdl=40.0) for tf in tf_grid
]



### Checkpoint 2


In [ ]:
test_bm25 = bm25_score(tf=3, df=2, N=10, dl=50, avgdl=40)
assert test_bm25 > 0, "Task 3: BM25 should be positive for present terms"
score_tf1 = bm25_score(tf=1, df=2, N=10, dl=50, avgdl=40)
score_tf10 = bm25_score(tf=10, df=2, N=10, dl=50, avgdl=40)
assert score_tf10 < 10 * score_tf1, "Task 3: BM25 should saturate"
print("[ok] Checkpoint 2 passed — BM25 with saturation + length normalisation\n")



## TASK 4 — VISUALISE: top-term ranking + BM25 saturation curve


In [ ]:
viz = ModelVisualizer()

# TODO: Compute the mean TF-IDF weight per column and surface the top 10.
# Hint: np.asarray(X_tfidf.mean(axis=0)).flatten().argsort()[-10:][::-1]
mean_tfidf = ____
top_idx = ____
top_terms = {tfidf_vocab[i]: float(mean_tfidf[i]) for i in top_idx}

term_ranking = {term: {"mean_tfidf": w} for term, w in top_terms.items()}
fig_terms = viz.metric_comparison(term_ranking)
fig_terms.update_layout(title="Top Terms by Mean TF-IDF")
fig_terms.write_html(str(OUTPUT_DIR / "ex6_1_tfidf_top_terms.html"))

sat_curve = {
    f"tf={tf}": {"BM25": float(score)} for tf, score in zip(tf_grid, saturation_scores)
}
fig_sat = viz.metric_comparison(sat_curve)
fig_sat.update_layout(title="BM25 Saturation Curve (k1=1.2, b=0.75)")
fig_sat.write_html(str(OUTPUT_DIR / "ex6_1_bm25_saturation.html"))



### Checkpoint 3


In [ ]:
assert len(top_terms) == 10, "Task 4: should surface top 10 terms"
assert saturation_scores[0] < saturation_scores[-1], "Task 4: BM25 monotonic in tf"
print("[ok] Checkpoint 3 passed — visualisations written\n")



## TASK 5 — APPLY: ST Engineering Internal Search


In [ ]:
print_scenario("tfidf_bm25")



## REFLECTION


[x] Built a bag-of-words and TF-IDF matrix with sklearn
  [x] Implemented BM25 with saturation + length normalisation
  [x] Visualised top terms and the BM25 saturation curve
  [x] Mapped the technique to ST Engineering internal search

  Next: 02_nmf_topics.py — factorise the TF-IDF matrix into topics.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

